# Pyannote Diarization testing - pyannote/speaker-diarization-community-1
# <https://huggingface.co/pyannote/speaker-diarization-community-1>

In [ ]:
# setup stuff

from pydantic_settings import BaseSettings, SettingsConfigDict
# from pydantic import BaseSettings

class NotebookSettings(BaseSettings):
  hf_token: str

settings = NotebookSettings()

print(f"HF_TOKEN env variable: {settings.hf_token}")


Looks like the `HF_TOKEN` environment variable is reading properly, huzzah, let's move on...

This pipeline ingests mono audio sampled at 16kHz and outputs speaker diarization.

 - stereo or multi-channel audio files are automatically downmixed to mono by averaging the channels.
 - audio files sampled at a different rate are resampled to 16kHz automatically upon loading.

The main improvements brought by Community-1 are:

 - improved speaker assignment and counting
 - simpler reconciliation with transcription timestamps with exclusive speaker diarization
 - easy offline use (i.e. without internet connection)
 - (optionally) hosted on pyannoteAI cloud


### Exclusive speaker diarization

Community-1 pre-trained pipeline returns a new exclusive speaker diarization, on top of the regular speaker diarization, available as `output.exclusive_speaker_diarization`.

This is a feature which is backported from our latest commercial model that simplifies the reconciliation between fine-grained speaker diarization timestamps and (sometimes not so precise) transcription timestamps.

In [ ]:
import hf

hf auth login


In [ ]:
# example usage from HF page use model dialog

from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-community-1")

# inference on the whole file
pipeline("file.wav")

# inference on an excerpt
from pyannote.core import Segment
excerpt = Segment(start = 2.0, end = 5.0)

from pyannote.audio import Audio
waveform, sample_rate = Audio().crop("file.wav", excerpt)
pipeline({"waveform": waveform, "sample_rate": sample_rate})


In [ ]:
# example usage from HF page itself

# download the pipeline from Huggingface
from pyannote.audio import Pipeline
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1", 
    token = settings.hf_token
    )

# run the pipeline locally on your computer
output = pipeline("audio.wav")

# print the predicted speaker diarization 
for turn, speaker in output.speaker_diarization:
    print(f"{speaker} speaks between t={turn.start:.3f}s and t={turn.end:.3f}s")


## Offline use

In the terminal, copy the pipeline on disk:


In [ ]:
# make sure git-lfs is installed (https://git-lfs.com)
git lfs install

# create a directory on disk
mkdir pipeline

# when prompted for a password, use an access token with write permissions.
# generate one from your settings: https://huggingface.co/settings/tokens
git clone https://hf.co/pyannote/speaker-diarization-community-1 pipeline/pyannote-speaker-diarization-community-1


In Python, use the pipeline without internet connection:


In [ ]:
# load pipeline from disk (works without internet connection)
from pyannote.audio import Pipeline
pipeline = Pipeline.from_pretrained("pipeline/pyannote-speaker-diarization-community-1")

# run the pipeline locally on your computer
output = pipeline("audio.wav")
